# 01 — LightGlue CUDA Graph Benchmark

Compare LightGlue (torch eager) vs CUDA Graph across batch sizes and point budgets.

**Key question:** How much does CUDA Graph remove kernel-launch overhead on Windows WDDM?

**Expected:** B=1/M=512/N=512: eager ~16ms → graph ~1.75ms (9× speedup)

**Dependencies:**
- A GPU with CUDA support (CUDA 11+)
- `accelerated_features` (for `LighterGlue` and `XFeatModel`)
- `kornia` (for LightGlue integration)
- `helper.py` from this repo

In [15]:
import sys, os, json, time, math
import numpy as np
import torch
import torch.nn.functional as F

# Relative path to project root (notebooks/ -> ../)
_PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath("__file__"))) if "__file__" in dir() else os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, _PROJECT_ROOT)

from helper import (
    patch_kornia_capture_safe,
    CGLightGlue,
    capture_cg_lightglue,
    replay_cg_lightglue,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")
if device == "cuda":
    print(f"GPU    = {torch.cuda.get_device_name(0)}")
    print(f"SM     = {torch.cuda.get_device_capability(0)}")

device = cuda
GPU    = NVIDIA GeForce RTX 4090 Laptop GPU
SM     = (8, 9)


In [16]:
# ── Build CGLightGlue model (einsum path, SDPA disabled) ──
patch_kornia_capture_safe()

model = CGLightGlue().to(device).eval()
MIN_CONF = model.net.conf.filter_threshold
print(f"LightGlue model loaded, n_layers={model.net.conf.n_layers}, "
      f"filter_threshold={MIN_CONF}")

# Quick smoke test
k0 = torch.zeros(1, 64, 2, device=device)
d0 = torch.zeros(1, 64, 64, device=device)
k1 = torch.zeros(1, 64, 2, device=device)
d1 = torch.zeros(1, 64, 64, device=device)
s0 = torch.tensor([[600, 400]], device=device, dtype=torch.long)
s1 = torch.tensor([[600, 400]], device=device, dtype=torch.long)
m0 = torch.zeros(1, 64, 1, device=device, dtype=torch.bool)
m1 = torch.zeros(1, 64, 1, device=device, dtype=torch.bool)
with torch.inference_mode():
    out = model(k0, d0, s0, k1, d1, s1, m0, m1, MIN_CONF)
print(f"Smoke test OK — output shapes: {[o.shape for o in out]}")

Loaded LightGlue model
LightGlue model loaded, n_layers=6, filter_threshold=0.1
Smoke test OK — output shapes: [torch.Size([1, 64]), torch.Size([1, 64]), torch.Size([1, 64]), torch.Size([1, 64])]


In [17]:
# ── Sweep parameters ──
BATCH_SIZES = [1, 2, 4, 8, 16]
POINT_BUDGETS = [64, 128, 256, 512]
WARM, REP = 10, 50

print(f"Sweep: B={BATCH_SIZES}  M=N={POINT_BUDGETS}")
print(f"Total configs: {len(BATCH_SIZES) * len(POINT_BUDGETS)}")

Sweep: B=[1, 2, 4, 8, 16]  M=N=[64, 128, 256, 512]
Total configs: 20


In [18]:
def make_dict_inputs(B, M, N, seed=42):
    """Generate B pairs of synthetic data as dicts."""
    np.random.seed(seed)
    d0_list, d1_list = [], []
    for b in range(B):
        n0 = np.random.randint(M // 2 + 1, M + 1) if M > 1 else 1
        n1 = np.random.randint(N // 2 + 1, N + 1) if N > 1 else 1
        d0_list.append({
            'keypoints': np.random.rand(n0, 2).astype(np.float32) * [600, 400],
            'descriptors': np.random.rand(n0, 64).astype(np.float32),
            'scores': np.random.rand(n0).astype(np.float32),
            'image_size': (640, 480),
        })
        d1_list.append({
            'keypoints': np.random.rand(n1, 2).astype(np.float32) * [600, 400],
            'descriptors': np.random.rand(n1, 64).astype(np.float32),
            'scores': np.random.rand(n1).astype(np.float32),
            'image_size': (640, 480),
        })
    return d0_list, d1_list


def dicts_to_tensors(d0_list, d1_list, B, M, N):
    """Convert dict format to eager batch tensors, sorting by score."""
    k0 = torch.zeros(B, M, 2, device=device)
    d0 = torch.zeros(B, M, 64, device=device)
    m0 = torch.zeros(B, M, 1, device=device, dtype=torch.bool)
    s0 = torch.zeros(B, 2, device=device, dtype=torch.long)
    k1 = torch.zeros(B, N, 2, device=device)
    d1 = torch.zeros(B, N, 64, device=device)
    m1 = torch.zeros(B, N, 1, device=device, dtype=torch.bool)
    s1 = torch.zeros(B, 2, device=device, dtype=torch.long)
    for b in range(B):
        n0 = min(len(d0_list[b]['scores']), M)
        n1 = min(len(d1_list[b]['scores']), N)
        if n0 > 0:
            idx = (-d0_list[b]['scores']).argsort()[:n0]
            k0[b, :n0] = torch.as_tensor(d0_list[b]['keypoints'][idx], device=device)
            d0[b, :n0] = torch.as_tensor(d0_list[b]['descriptors'][idx], device=device)
            m0[b, :n0, 0] = True
        s0[b] = torch.tensor(d0_list[b]['image_size'], dtype=torch.long, device=device)
        if n1 > 0:
            idx = (-d1_list[b]['scores']).argsort()[:n1]
            k1[b, :n1] = torch.as_tensor(d1_list[b]['keypoints'][idx], device=device)
            d1[b, :n1] = torch.as_tensor(d1_list[b]['descriptors'][idx], device=device)
            m1[b, :n1, 0] = True
        s1[b] = torch.tensor(d1_list[b]['image_size'], dtype=torch.long, device=device)
    return k0, d0, s0, k1, d1, s1, m0, m1

In [19]:
# ── Main benchmark loop ──
# Compare batch CUDA Graph vs batch eager (M=N only)
results = []
total = len(BATCH_SIZES) * len(POINT_BUDGETS)
done = 0

for B in BATCH_SIZES:
    for M in POINT_BUDGETS:
        N = M
        done += 1
        seed = 42 + B * 10000 + M

        # Build model + capture batch CUDA Graph
        model = CGLightGlue().to(device).eval()
        cg = capture_cg_lightglue(model, B=B, M=M, N=N, device=device)

        # Generate data once per config
        d0_list, d1_list = make_dict_inputs(B, M, N, seed=seed)
        tensors = dicts_to_tensors(d0_list, d1_list, B, M, N)

        # Eager batch timing
        with torch.inference_mode():
            for _ in range(WARM):
                model(*tensors, MIN_CONF)
            torch.cuda.synchronize()
            el = []
            for _ in range(REP):
                t0 = time.perf_counter()
                model(*tensors, MIN_CONF)
                torch.cuda.synchronize()
                el.append((time.perf_counter() - t0) * 1000.0)
        el = np.array(el)

        # Batch CUDA Graph timing
        with torch.inference_mode():
            for _ in range(WARM):
                replay_cg_lightglue(cg, d0_list, d1_list, 0.1)
            torch.cuda.synchronize()
            gl = []
            for _ in range(REP):
                t0 = time.perf_counter()
                replay_cg_lightglue(cg, d0_list, d1_list, 0.1)
                torch.cuda.synchronize()
                gl.append((time.perf_counter() - t0) * 1000.0)
        gl = np.array(gl)

        rec = {
            "B": B, "M": M, "N": N,
            "eager_ms_p50": float(np.percentile(el, 50)),
            "eager_ms_mean": float(el.mean()),
            "graph_ms_p50": float(np.percentile(gl, 50)),
            "graph_ms_mean": float(gl.mean()),
            "speedup_p50": float(np.percentile(el, 50) / np.percentile(gl, 50)),
        }
        results.append(rec)

        del cg, model
        torch.cuda.empty_cache()

        print(f"[{done}/{total}] B={B} M={M} N={N}:  "
              f"eager={rec['eager_ms_p50']:.2f}ms  "
              f"graph={rec['graph_ms_p50']:.2f}ms  "
              f"speedup={rec['speedup_p50']:.2f}x", flush=True)

Loaded LightGlue model
[1/20] B=1 M=64 N=64:  eager=22.29ms  graph=2.68ms  speedup=8.33x
Loaded LightGlue model
[2/20] B=1 M=128 N=128:  eager=24.20ms  graph=2.49ms  speedup=9.72x
Loaded LightGlue model
[3/20] B=1 M=256 N=256:  eager=20.03ms  graph=2.27ms  speedup=8.82x
Loaded LightGlue model
[4/20] B=1 M=512 N=512:  eager=19.38ms  graph=3.25ms  speedup=5.96x
Loaded LightGlue model
[5/20] B=2 M=64 N=64:  eager=20.39ms  graph=3.50ms  speedup=5.83x
Loaded LightGlue model
[6/20] B=2 M=128 N=128:  eager=19.02ms  graph=3.22ms  speedup=5.91x
Loaded LightGlue model
[7/20] B=2 M=256 N=256:  eager=18.12ms  graph=5.36ms  speedup=3.38x
Loaded LightGlue model
[8/20] B=2 M=512 N=512:  eager=19.02ms  graph=6.78ms  speedup=2.81x
Loaded LightGlue model
[9/20] B=4 M=64 N=64:  eager=19.02ms  graph=5.52ms  speedup=3.45x
Loaded LightGlue model
[10/20] B=4 M=128 N=128:  eager=17.89ms  graph=5.95ms  speedup=3.01x
Loaded LightGlue model
[11/20] B=4 M=256 N=256:  eager=18.67ms  graph=6.87ms  speedup=2.72x
Loa

---
## Results summary

In [20]:
# ── Results summary ──
print("\n" + "=" * 70)
print("SUMMARY: LightGlue CUDA Graph vs Eager")
print("=" * 70)
print(f"{'B':>3} {'M':>4} {'N':>4}  {'eager(ms)':>10} {'graph(ms)':>10} {'speedup':>8}")
print("-" * 45)
for r in results:
    print(f"{r['B']:>3} {r['M']:>4} {r['N']:>4}  "
          f"{r['eager_ms_p50']:>8.2f}  {r['graph_ms_p50']:>8.2f}  "
          f"{r['speedup_p50']:>6.2f}x")


SUMMARY: LightGlue CUDA Graph vs Eager
  B    M    N   eager(ms)  graph(ms)  speedup
---------------------------------------------
  1   64   64     22.29      2.68    8.33x
  1  128  128     24.20      2.49    9.72x
  1  256  256     20.03      2.27    8.82x
  1  512  512     19.38      3.25    5.96x
  2   64   64     20.39      3.50    5.83x
  2  128  128     19.02      3.22    5.91x
  2  256  256     18.12      5.36    3.38x
  2  512  512     19.02      6.78    2.81x
  4   64   64     19.02      5.52    3.45x
  4  128  128     17.89      5.95    3.01x
  4  256  256     18.67      6.87    2.72x
  4  512  512     17.89      8.77    2.04x
  8   64   64     18.40      7.25    2.54x
  8  128  128     20.59      8.05    2.56x
  8  256  256     18.27      9.48    1.93x
  8  512  512     17.84     14.89    1.20x
 16   64   64     18.89     11.44    1.65x
 16  128  128     18.45     13.44    1.37x
 16  256  256     18.36     16.67    1.10x
 16  512  512     18.16     25.72    0.71x


In [21]:
# ── Best / worst cases ──
by_spd = sorted(results, key=lambda r: r['speedup_p50'])
print("\nWorst 3 speedups:")
for r in by_spd[:3]:
    print(f"  B={r['B']} M={r['M']} N={r['N']}:  "
          f"{r['eager_ms_p50']:.2f}ms → {r['graph_ms_p50']:.2f}ms  ({r['speedup_p50']:.2f}x)")

print("\nBest 3 speedups:")
for r in reversed(by_spd[-3:]):
    print(f"  B={r['B']} M={r['M']} N={r['N']}:  "
          f"{r['eager_ms_p50']:.2f}ms → {r['graph_ms_p50']:.2f}ms  ({r['speedup_p50']:.2f}x)")

print(f"\nFixed M=N=512, vary B:")
print(f"{'B':>3}  {'eager(ms)':>8} {'graph(ms)':>8} {'speedup':>8}")
for r in results:
    if r['M'] == 512 and r['N'] == 512:
        print(f"{r['B']:>3}  {r['eager_ms_p50']:>8.2f}  {r['graph_ms_p50']:>8.2f}  {r['speedup_p50']:>6.2f}x")


Worst 3 speedups:
  B=16 M=512 N=512:  18.16ms → 25.72ms  (0.71x)
  B=16 M=256 N=256:  18.36ms → 16.67ms  (1.10x)
  B=8 M=512 N=512:  17.84ms → 14.89ms  (1.20x)

Best 3 speedups:
  B=1 M=128 N=128:  24.20ms → 2.49ms  (9.72x)
  B=1 M=256 N=256:  20.03ms → 2.27ms  (8.82x)
  B=1 M=64 N=64:  22.29ms → 2.68ms  (8.33x)

Fixed M=N=512, vary B:
  B  eager(ms) graph(ms)  speedup
  1     19.38      3.25    5.96x
  2     19.02      6.78    2.81x
  4     17.89      8.77    2.04x
  8     17.84     14.89    1.20x
 16     18.16     25.72    0.71x
